In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Project setup
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If the notebook is being run from the notebooks folder,
# move one level up to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", ANALYSIS_DIR)

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load and prepare movie data
# --------------------------------------------------

ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

movies = pd.read_csv(
    ANALYSIS_DIR / "movies.csv"
)

movies["release_year"] = pd.to_numeric(
    movies["release_year"], errors="coerce"
)

movies["budget"] = pd.to_numeric(
    movies["budget"], errors="coerce"
)

movies["revenue"] = pd.to_numeric(
    movies["revenue"], errors="coerce"
)

# Do not silently treat missing or invalid financial values as zero.
movies["profit"] = movies["revenue"] - movies["budget"]

movies["movie_roi_pct"] = np.where(
    movies["budget"] > 0,
    (movies["profit"] / movies["budget"]) * 100,
    np.nan
)

movies["revenue_multiple"] = np.where(
    movies["budget"] > 0,
    movies["revenue"] / movies["budget"],
    np.nan
)


# --------------------------------------------------
# 2. Classify licensing units within each period
# --------------------------------------------------

def build_licensing_units(period_movies, include_single_collections=False):
    period_movies = period_movies.copy()

    collection_sizes = (
        period_movies.dropna(subset=["collection_id"])
        .groupby("collection_id")["tmdb_id"]
        .nunique()
    )

    period_movies["period_collection_count"] = (
        period_movies["collection_id"].map(collection_sizes)
    )

    period_movies["licensing_type"] = np.select(
        [
            period_movies["collection_id"].isna(),
            period_movies["period_collection_count"].ge(2),
            (
                include_single_collections
                & period_movies["period_collection_count"].eq(1)
            )
        ],
        [
            "Standalone film",
            "Franchise bundle",
            "Standalone film"
        ],
        default="Single-film collection candidate"
    )

    comparable = period_movies[
        period_movies["licensing_type"].isin(
            ["Franchise bundle", "Standalone film"]
        )
    ].copy()

    comparable["licensing_unit_id"] = np.where(
        comparable["licensing_type"] == "Franchise bundle",
        "collection_" + comparable["collection_id"].astype(str),
        "movie_" + comparable["tmdb_id"].astype(str)
    )

    units = (
        comparable.groupby(
            ["licensing_type", "licensing_unit_id"],
            as_index=False
        )
        .agg(
            titles=("title", lambda x: " | ".join(x)),
            film_count=("tmdb_id", "nunique"),
            total_budget=("budget", "sum"),
            total_revenue=("revenue", "sum"),
            total_profit=("profit", "sum"),
            median_film_revenue=("revenue", "median"),
            median_film_budget=("budget", "median")
        )
    )

    units["gross_revenue_less_budget"] = (
        units["total_revenue"] - units["total_budget"]
    )

    units["gross_revenue_to_budget_pct"] = np.where(
        units["total_budget"] > 0,
        units["gross_revenue_less_budget"]
        / units["total_budget"] * 100,
        np.nan
    )

    units["revenue_multiple"] = np.where(
        units["total_budget"] > 0,
        units["total_revenue"] / units["total_budget"],
        np.nan
    )

    return units, period_movies


# --------------------------------------------------
# 3. Build the two analysis periods
# --------------------------------------------------

periods = {
    "All release years": movies.copy(),
    "2000 onward": movies[
        movies["release_year"] >= 2000
    ].copy()
}

unit_datasets = {}
sensitivity_datasets = {}

for period_name, period_movies in periods.items():

    # Primary analysis: exclude single-film collection candidates.
    units, classified = build_licensing_units(
        period_movies,
        include_single_collections=False
    )

    unit_datasets[period_name] = units

    # Sensitivity analysis: treat single-film collection candidates
    # as standalone licensing opportunities.
    sensitivity_units, _ = build_licensing_units(
        period_movies,
        include_single_collections=True
    )

    sensitivity_datasets[period_name] = sensitivity_units

    print(f"\n{period_name}")
    print(classified["licensing_type"].value_counts())


# --------------------------------------------------
# 4. Compare licensing performance
# --------------------------------------------------

def compare_units(unit_data, analysis_label):

    results = []

    for period_name, units in unit_data.items():

        for licensing_type, group in units.groupby("licensing_type"):

            total_budget = group["total_budget"].sum()
            total_revenue = group["total_revenue"].sum()
            total_profit = group["total_profit"].sum()

            results.append({
                "analysis": analysis_label,
                "period": period_name,
                "licensing_type": licensing_type,
                "licensing_units": len(group),
                "films": group["film_count"].sum(),
                "total_budget": total_budget,
                "total_revenue": total_revenue,
                "gross_revenue_less_budget": total_profit,
                "weighted_gross_return_pct": (
                    total_profit / total_budget * 100
                    if total_budget > 0 else np.nan
                ),
                "median_unit_gross_return_pct": (
                    group["gross_revenue_to_budget_pct"].median()
                ),
                "median_revenue_multiple": (
                    group["revenue_multiple"].median()
                )
            })

    return pd.DataFrame(results)


primary_comparison = compare_units(
    unit_datasets,
    "Primary: multi-film bundles only"
)

sensitivity_comparison = compare_units(
    sensitivity_datasets,
    "Sensitivity: single-film collections treated as standalone"
)

comparison_df = pd.concat(
    [primary_comparison, sensitivity_comparison],
    ignore_index=True
)

print("\nLICENSING PERFORMANCE COMPARISON")
print(comparison_df.round(2).to_string(index=False))


# --------------------------------------------------
# 5. Save corrected outputs
# --------------------------------------------------

movies.to_csv(
    ANALYSIS_DIR / "movie_financial_metrics.csv",
    index=False
)

comparison_df.to_csv(
    ANALYSIS_DIR / "licensing_comparison.csv",
    index=False
)

for period_name, units in unit_datasets.items():

    filename = (
        "licensing_units_all_years.csv"
        if period_name == "All release years"
        else "licensing_units_2000_onward.csv"
    )

    units.to_csv(ANALYSIS_DIR / filename, index=False)

print("\nCorrected comparison outputs saved.")

# --------------------------------------------------
# 5. Save analysis outputs
# --------------------------------------------------

movies.to_csv(
    ANALYSIS_DIR / "movie_financial_metrics.csv",
    index=False
)

comparison_df.to_csv(
    ANALYSIS_DIR / "licensing_comparison.csv",
    index=False
)

for period_name, units in unit_datasets.items():
    file_name = (
        "licensing_units_all_years.csv"
        if period_name == "All release years"
        else "licensing_units_2000_onward.csv"
    )

    units.to_csv(
        ANALYSIS_DIR / file_name,
        index=False
    )

print("\nANALYSIS OUTPUTS SAVED")
print("movie_financial_metrics.csv")
print("licensing_comparison.csv")
print("licensing_units_all_years.csv")
print("licensing_units_2000_onward.csv")

# --------------------------------------------------
# 6. Basic integrity checks
# --------------------------------------------------

assert movies["tmdb_id"].is_unique
assert comparison_df["period"].nunique() == 2
assert set(comparison_df["licensing_type"]) == {
    "Franchise bundle",
    "Standalone film"
}

print("\nAll financial-analysis integrity checks passed.")


Project root: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis
Processed data directory: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\data\processed

All release years
licensing_type
Franchise bundle                    87
Standalone film                     16
Single-film collection candidate     9
Name: count, dtype: int64

2000 onward
licensing_type
Franchise bundle                    56
Standalone film                      8
Single-film collection candidate     7
Name: count, dtype: int64

LICENSING PERFORMANCE COMPARISON
                                                  analysis            period   licensing_type  licensing_units  films  total_budget  total_revenue  gross_revenue_less_budget  weighted_gross_return_pct  median_unit_gross_return_pct  median_revenue_multiple
                          Primary: multi-film bundles only All release years Franchise bundle               23     87   10643250000    58768179240                48124929240      

In [3]:
print(
    comparison_df[
        comparison_df["analysis"].str.startswith("Sensitivity")
    ].round(2).to_string(index=False)
)

                                                  analysis            period   licensing_type  licensing_units  films  total_budget  total_revenue  gross_revenue_less_budget  weighted_gross_return_pct  median_unit_gross_return_pct  median_revenue_multiple
Sensitivity: single-film collections treated as standalone All release years Franchise bundle               23     87   10643250000    58768179240                48124929240                     452.16                        455.11                     5.55
Sensitivity: single-film collections treated as standalone All release years  Standalone film               25     25    1502700000     8393602523                 6890902523                     458.57                        400.00                     5.00
Sensitivity: single-film collections treated as standalone       2000 onward Franchise bundle               17     56    8968000000    46195146097                37227146097                     415.11                        361.15  

In [4]:
print("\nSINGLE-FILM COLLECTION CANDIDATES")

for period_name, period_movies in periods.items():
    collection_sizes = (
        period_movies.dropna(subset=["collection_id"])
        .groupby("collection_id")["tmdb_id"]
        .nunique()
    )

    candidates = period_movies[
        period_movies["collection_id"].notna()
        & period_movies["collection_id"].map(collection_sizes).eq(1)
    ]

    print(f"\n{period_name}:")
    print(
        candidates[["title", "release_year", "collection_name"]]
        .to_string(index=False)
    )


SINGLE-FILM COLLECTION CANDIDATES

All release years:
                   title  release_year                    collection_name
            Blade Runner          1982            Blade Runner Collection
The Silence of the Lambs          1991     The Hannibal Lecter Collection
                    Jaws          1975                    Jaws Collection
          Monsters, Inc.          2001          Monsters, Inc. Collection
            Ghostbusters          1984            Ghostbusters Collection
           The Lion King          1994           The Lion King Collection
         The Incredibles          2004         The Incredibles Collection
              The Smurfs          2011 The Smurfs (Theatrical) Collection
      Mad Max: Fury Road          2015                 Mad Max Collection

2000 onward:
                                             title  release_year                    collection_name
Indiana Jones and the Kingdom of the Crystal Skull          2008           Indiana Jones Co